In [ ]:
import os
import shutil
import subprocess

mach_numbers = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
radii = [50, 100, 200]
grids = {
    'c': '20, 20', 
    'm': '40, 40', 
    'f': '60, 60'  
}

time = {
    'c': '125 0.02',
    'm': '250 0.01',
    'f': '500 0.005'
}

speed_of_sound = 340.0 # m/s (170 / 0.5)

config_path = 'input_config.txt'
source_path = 'input_source.txt'
default_solver_output = 'FWH_observer_time_01_RT.dat' 
output_dir = 'Data 260527/test_mach/'

os.makedirs(output_dir, exist_ok=True)

# 2. File Editing Helper
def edit_line(filepath, line_idx, new_value):
    """Directly overwrites a specific line index."""
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    lines[line_idx] = f"{new_value}\n"
    
    with open(filepath, 'w') as f:
        f.writelines(lines)


for M in mach_numbers:

    ux = M * speed_of_sound
    source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
    edit_line(source_path, 14, source_val) 
    
    m_str = str(M).replace('.', '')
    
    # for r in radii:
    #     for grid_label, grid_res in grids.items():
            
    #         # Inject radius and grid into config 
    #         config_val = f"{r}, {grid_res}"
    #         edit_line(config_path, 6, config_val) 
            
    #         print(f"Running Case: R={r} | M={M} | Grid={grid_label.upper()}")
            
    #         # Execute the compiled Fortran binary
    #         subprocess.run(["../../bin/FWH_Sphere"], check=True) 
            
    #         print(f"Renaming and moving file r{r}_M{m_str}{grid_label}.dat")
    #         new_filename = f"r{r}_M{m_str}{grid_label}.dat"
    #         destination = os.path.join(output_dir, new_filename)
            
    #         shutil.move(default_solver_output, destination)

    for time_label, time_res in time.items():
        config_val = f"{time_res}"
        edit_line(config_path, 14, config_val)

        print(f'Running Case: timestep size={time_res} | M={M}')
        # subprocess.run(["../../bin/FWH_Sphere"], check=True) 

#         # Generate the new filename: r200_M09f.dat
        print(f"Renaming and moving file t_M{m_str}{time_label}.dat")
        new_filename = f"t_M{m_str}{time_label}.dat"
        destination = os.path.join(output_dir, new_filename)
        # shutil.move(default_solver_output, destination)

print("Batch processing complete.")

Running Case: timestep size=125 0.02 | M=0.5
 ... Initializing MPI
 ... MPI initialized successfully
 ... Getting MPI rank and size
 ... Total number of MPI processes: 1
 ... MPI rank of current process: 0
 ... Reading config parameters

 ... reading input parameters from file

 FW-H parameters: 
 Far-field sound speed        :  340.00000 [m/s]
 Far-field air density        :    1.20000 [kg/m^3]
 FW-H surface grid type       : SPHERE               
 FW-H surface grid radius     :   50.00000 [m]
 FW-H surface grid lines      :         60        60 
 FW-H surface grid motion     : SOURCE               
 FW-H surface data type       : SOURCE               
 FW-H integration method      : RETARDED_TIME        
 FW-H kernel type             : TIME_DOMAIN          
 FW-H time integration levels :        125 
 FW-H time stepping           :    0.02000 [s]
 Temporal DFT post-processing : F 
 Debugging level              :          0 

 Analytical source definitions: 
 Source #01:
   Type      

In [ ]:
import os
import shutil
import subprocess
import numpy as np

mach_numbers = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
cfl_targets = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]

X_target = 20.0        
R = 20.0              
speed_of_sound = 340.0
omega = 10.0
f_source = omega / (2 * np.pi)
T_sim = 2.5            

config_path = 'input_config.txt'
source_path = 'input_source.txt'
default_solver_output = 'FWH_observer_time_01_RT.dat' 
output_dir = 'data_260528/test_cfl/'

os.makedirs(output_dir, exist_ok=True)

def edit_line(filepath, line_idx, new_value):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    lines[line_idx] = f"{new_value}\n"
    with open(filepath, 'w') as f:
        f.writelines(lines)

for M in mach_numbers:
    
    ux = M * speed_of_sound
    source_val = f"{ux}, 0.0, 0.0, 0.0, 0.0, 0.0"
    edit_line(source_path, 14, source_val) 
    
    m_str = f"{int(M * 100):02d}" 
    
    # dx = c(1-M) / (X * f_s)
    dx = speed_of_sound * (1 - M) / (X_target * f_source)
    
    # N = 2*pi*R / dx
    N = int(np.round((2 * np.pi * R) / dx))
    
    config_val_spatial = f"{R}, {N}, {N}" 
    edit_line(config_path, 6, config_val_spatial) 
    
    for cfl in cfl_targets:
        # P = X / CFL
        P_target = X_target / cfl
        
        # Calculate dynamic dt: dt = (1-M) / (P * f_s)
        dt = (1 - M) / (P_target * f_source)
        
        # Calculate total steps required to hit 2.5 seconds
        total_steps = int(np.ceil(T_sim / dt))
        
        # Inject total_steps and dt (Formatting dt to 6 decimal places for Fortran safety)
        config_val_time = f"{total_steps} {dt:.6f}"
        edit_line(config_path, 14, config_val_time)
        
        print(f"Running Case: M={M:.2f} | CFL={cfl} (X={X_target}, P={P_target:.1f}) | N={N} | dt={dt:.6f}")
        
        cfl_str = str(cfl).replace('.', '')
        
        # --- UNCOMMENT BELOW TO ACTUALLY RUN THE SOLVER ---
        
        # subprocess.run(["mpirun", "-np", "2", "../bin/FWH_Sphere"], check=True)
        
        # new_filename = f"cfl{cfl_str}_M{m_str}.dat"
        # destination = os.path.join(output_dir, new_filename)
        # shutil.move(default_solver_output, destination)

print("Batch processing complete.")

Running Case: M=0.50 | CFL=0.25 (X=20.0, P=80.0) | N=24 | dt=0.003927
 ... Initializing MPI
 ... Initializing MPI
 ... MPI initialized successfully
 ... Getting MPI rank and size
 ... Total number of MPI processes: 2
 ... MPI rank of current process: 0
 ... Reading config parameters

 ... reading input parameters from file
 ... MPI initialized successfully
 ... Getting MPI rank and size
 ... Total number of MPI processes: 2
 ... MPI rank of current process: 1
 ... Reading config parameters

 ... reading input parameters from file

 FW-H parameters: 

 FW-H parameters: 
 Far-field sound speed        :  340.00000 [m/s]
 Far-field air density        :    1.20000 [kg/m^3]
 FW-H surface grid type       : SPHERE               
 FW-H surface grid radius     :   20.00000 [m]
 FW-H surface grid lines      :         24        24 
 FW-H surface grid motion     : SOURCE               
 FW-H surface data type       : SOURCE               
 FW-H integration method      : RETARDED_TIME        
 FW-H 

In [ ]:
def edit_config(input_type, path, variable, value):
    config_var_dict = {'c,rho':2, 'radius, grid':6, 'total_timestep, timestep_size':14}
    source_var_dict = {'freq,strength,phase':6, 'x,y,z':10, 'ux,uy,uz,rx,ry,rz':14}
    observer_var_dict  = {'x,y,z':6, 'motion_type, duration':8, 'ux,uy,uz,rx,ry,rz':10}
    
    with open(path, 'r') as f:
        lines = f.readlines()
    
    if input_type=='config' and variable in config_var_dict:
        lines[config_var_dict[variable]] = f'{value}\n'
    elif input_type=='source' and variable in source_var_dict:
        lines[source_var_dict[variable]] = f'{value}\n'
    elif input_type=='observer' and variable in observer_var_dict:
        lines[observer_var_dict[variable]] = f'{value}\n'
        
    with open(path, 'w') as f:
        f.writelines(lines)